# TensorFlow

A refresher on **TensorFlow** — Google's end-to-end machine-learning platform: a tensor + autodiff engine (`tf.GradientTape`), a graph compiler (`tf.function`), an input pipeline (`tf.data`), and a production deployment story (TF Serving, TFLite, TF.js) — with **Keras** as its high-level model API.

**Domain:** AI/ML Tooling  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**TensorFlow is a numerical-computing library built around tensors (n-dimensional arrays) with automatic differentiation and hardware acceleration (CPU/GPU/TPU).** On top of those primitives it ships everything you need to train and *ship* a model: [Keras](keras.ipynb) for high-level model building (`model.fit`), `tf.data` for input pipelines, `tf.function` to compile Python into fast graphs, and a deployment family — **TF Serving** (servers), **TFLite/LiteRT** (mobile & edge), **TF.js** (browser).

**The problem it solves.** Training a model means computing gradients of a scalar loss w.r.t. millions of parameters, then doing it *fast* across accelerators, then running the result reliably in production. TensorFlow records your math, differentiates it for you (`tf.GradientTape`), compiles hot paths into optimized graphs (`tf.function`), and gives you a battle-tested serving/edge runtime so the trained artifact runs the same off your laptop.

**Why it matters.** TF was the first framework to make industrial-scale training and deployment mainstream, and it still owns much of the *production* and *on-device* story: TPUs are a first-class target, TFLite runs on billions of phones, and TF Serving is a mature model server. Since **TF 2.x** it runs **eager by default** (like NumPy/PyTorch), so the old "build a static graph, then run a session" ceremony is gone — you get graphs only when you opt in with `tf.function`.

**Reach for it when** you need a hardened deployment path (mobile/edge/browser/serving), you're targeting **TPUs**, you're in a Google-ecosystem stack, or you want Keras's batteries-included training loop with TF's runtime underneath.

**Don't reach for it when** you're doing research that lives on the PyTorch/Hugging Face ecosystem (most open-weights models ship as [PyTorch](pytorch.ipynb)), or classical ML on tabular data fits ([scikit-learn](scikit-learn.ipynb)). For functional/JIT-first or TPU research, [JAX/Flax](jax-flax.ipynb) is the modern alternative.

## 2. Mental Model

**A `tf.Tensor` is an immutable NumPy array on an accelerator; `tf.GradientTape` is a camcorder that records the math so it can be replayed backwards.**

In eager mode, ops run immediately and return concrete values — exactly like NumPy. To get gradients you wrap the forward pass in a tape, which *records* every op touching a watched tensor. Calling `tape.gradient(loss, vars)` replays that recording in reverse (the chain rule) to produce gradients:

```
with tf.GradientTape() as tape:     # ◀ recording starts
    pred = model(x)                 #   ops are taped
    loss = loss_fn(y, pred)         #   ◀ recording stops at end of block
grads = tape.gradient(loss, model.trainable_variables)   # replay in reverse
optimizer.apply_gradients(zip(grads, model.trainable_variables))
```

Two layers sit on top of this loop:

- **`tf.function`** wraps a Python function and, on first call, *traces* it into a static dataflow graph (a `ConcreteFunction`). Later calls skip the Python interpreter and run the compiled graph — faster, and portable to serving/TPU. You write eager Python; you opt into graph speed.
- **Keras** wraps the whole forward/loss/tape/apply loop into `model.fit()`, so you usually *don't* write the tape by hand — but it's running underneath, and you drop down to it for custom training.

Mantra: **eager to debug, `tf.function` to go fast, Keras to skip the boilerplate.**

## 3. Key Concepts

- **`tf.Tensor`** — the core value: an **immutable** n-d array with a `dtype` and `shape`, living on a device. NumPy-like (broadcasting, slicing); `.numpy()` pulls it back to a NumPy array.
- **`tf.Variable`** — the *mutable* counterpart: holds trainable state (weights). Optimizers update variables in place via `.assign(...)`; the tape watches them automatically.
- **`tf.GradientTape`** — the reverse-mode autodiff engine. Records ops on watched tensors/variables inside its `with` block; `tape.gradient(target, sources)` returns the gradients. Persistent tapes allow multiple `gradient()` calls.
- **`tf.function`** — `@tf.function` traces a Python function into a graph for speed and portability. **Tracing** happens once per distinct input *signature*; watch for accidental re-tracing (a big perf trap).
- **Eager vs graph execution** — TF 2.x runs eager by default (immediate, debuggable). `tf.function` is graph (compiled, fast, no Python in the loop). Same code, two modes.
- **Keras (`tf.keras`)** — the high-level API: `layers`, `Model`/`Sequential`, `compile()`, `fit()`, `evaluate()`, `predict()`. The default way to build models; Keras 3 is multi-backend but `tf.keras` targets the TF runtime. See [Keras](keras.ipynb).
- **`tf.data.Dataset`** — the input pipeline: `.map()`, `.batch()`, `.shuffle()`, `.prefetch()` build a streaming, parallel, GPU-feeding data loader that doesn't blow up memory.
- **Optimizers & losses** — `tf.keras.optimizers` (SGD, Adam) own the update rule; `tf.keras.losses` provide the scalar objective the tape differentiates.
- **`SavedModel`** — TF's language-neutral serialization format (graph + weights + signatures). The canonical artifact for TF Serving / TFLite. Keras also has its own `.keras` file format.
- **Deployment family** — **TF Serving** (server), **TFLite / LiteRT** (mobile/edge), **TF.js** (browser). The reason teams pick TF when shipping matters.

## 4. Setup

TensorFlow is `pip`-installable and the standard wheel includes GPU support on Linux (CUDA bundled). On macOS, the base `tensorflow` package runs on CPU; add `tensorflow-metal` for Apple-GPU acceleration.

```bash
# CPU / standard (what this notebook uses — portable, no GPU needed):
pip install tensorflow

# Apple Silicon GPU acceleration (optional):
pip install tensorflow tensorflow-metal
```

In a notebook you'd run `%pip install tensorflow`. The cell below imports TF (quieting its startup logs), reports the version and devices, and shows the eager, NumPy-like nature of a `tf.Tensor`.

In [1]:
import os

# Quiet TF's verbose C++ startup logging (0=all … 3=errors only). Set BEFORE import.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import tensorflow as tf

tf.random.set_seed(0)  # reproducible RNG for this notebook

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {tf.keras.__version__}")
print(f"Eager execution    : {tf.executing_eagerly()}")
print(f"Visible devices    : {[d.device_type for d in tf.config.list_physical_devices()]}")

# A tf.Tensor is an immutable, NumPy-like array that knows its dtype and shape.
x = tf.constant([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]])
print("\nTensor x:")
print(x)
print(f"shape={tuple(x.shape)}  dtype={x.dtype.name}")
print("sum via TF :", tf.reduce_sum(x).numpy())  # .numpy() bridges back to NumPy
print("as NumPy   :", x.numpy().sum())

TensorFlow version : 2.21.0
Keras version      : 3.14.1
Eager execution    : True
Visible devices    : ['CPU']

Tensor x:
tf.Tensor(
[[0. 1. 2.]
 [3. 4. 5.]], shape=(2, 3), dtype=float32)
shape=(2, 3)  dtype=float32
sum via TF : 15.0
as NumPy   : 15.0


## 5. Worked Examples

Three self-contained, CPU-friendly examples that build on each other:

1. **Autodiff with `GradientTape`** — get a gradient for free and check it against calculus.
2. **A custom training loop** — recover a known line with the explicit tape → gradients → `apply_gradients` loop (the TF-idiomatic low-level training step).
3. **`tf.function` + `tf.data`** — compile the step into a graph and stream batches through a pipeline.

### Example 1 — `GradientTape`: gradients for free

Take `y = x²·sin(x)` at `x = 2.0`. By calculus `dy/dx = 2x·sin(x) + x²·cos(x)`. The tape should match it without us ever writing the derivative.

In [2]:
import math

x = tf.Variable(2.0)  # a Variable is watched by the tape automatically

with tf.GradientTape() as tape:      # ◀ start recording
    y = x**2 * tf.sin(x)             #   every op on x is taped

dy_dx = tape.gradient(y, x)          # replay in reverse → gradient

manual = 2 * 2.0 * math.sin(2.0) + 2.0**2 * math.cos(2.0)

print(f"y                  = {y.numpy():.6f}")
print(f"dy/dx (tape)       = {dy_dx.numpy():.6f}")
print(f"manual derivative  = {manual:.6f}")
print(f"match: {math.isclose(dy_dx.numpy(), manual, rel_tol=1e-5)}")

y                  = 3.637190
dy/dx (tape)       = 1.972602
manual derivative  = 1.972602
match: True


### Example 2 — A custom training loop

Generate data from a known line `y = 3x + 2` (plus noise), then let TF *recover* the slope and intercept by gradient descent. This is the canonical TF training step written out by hand: **record forward pass on the tape → `tape.gradient` → `optimizer.apply_gradients`**. (Keras's `model.fit` wraps exactly this.)

In [3]:
tf.random.set_seed(0)

# Synthetic data from a known relationship: y = 3x + 2 + noise.
X = tf.reshape(tf.linspace(-1.0, 1.0, 100), (100, 1))
true_w, true_b = 3.0, 2.0
y = true_w * X + true_b + 0.1 * tf.random.normal((100, 1))

# Trainable parameters live in tf.Variables.
w = tf.Variable(0.0)
b = tf.Variable(0.0)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.1)

for epoch in range(200):
    with tf.GradientTape() as tape:
        pred = w * X + b                     # forward pass (recorded)
        loss = tf.reduce_mean((pred - y) ** 2)  # mean squared error
    grads = tape.gradient(loss, [w, b])       # d(loss)/d(params)
    optimizer.apply_gradients(zip(grads, [w, b]))  # update in place

    if epoch % 50 == 0 or epoch == 199:
        print(f"epoch {epoch:3d}  loss={loss.numpy():.4f}")

print(f"\nlearned: y = {w.numpy():.3f}x + {b.numpy():.3f}   (target: y = {true_w}x + {true_b})")

epoch   0  loss=7.0529
epoch  50  loss=0.0129
epoch 100  loss=0.0102
epoch 150  loss=0.0102
epoch 199  loss=0.0102

learned: y = 3.002x + 1.995   (target: y = 3.0x + 2.0)


### Example 3 — `tf.function` (graph speed) + `tf.data` (input pipeline)

`@tf.function` traces a Python function into a compiled graph on first call; later calls skip the Python interpreter. `tf.data.Dataset` builds a streaming, batched, prefetching input pipeline. Here we mini-batch the same regression and confirm the compiled step still learns the line.

In [4]:
tf.random.set_seed(0)
w = tf.Variable(0.0)
b = tf.Variable(0.0)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.1)

# tf.data: shuffle + batch into a streaming pipeline; prefetch overlaps prep with compute.
dataset = (
    tf.data.Dataset.from_tensor_slices((X, y))
    .shuffle(100)
    .batch(16)
    .prefetch(tf.data.AUTOTUNE)
)

# @tf.function compiles this step into a graph the first time it runs.
@tf.function
def train_step(xb, yb):
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean((w * xb + b - yb) ** 2)
    grads = tape.gradient(loss, [w, b])
    optimizer.apply_gradients(zip(grads, [w, b]))
    return loss

for epoch in range(30):
    for xb, yb in dataset:
        loss = train_step(xb, yb)

print(f"concrete functions traced: {len(train_step._list_all_concrete_functions())}")
print(f"final batch loss : {loss.numpy():.4f}")
print(f"learned: y = {w.numpy():.3f}x + {b.numpy():.3f}   (target: y = {true_w}x + {true_b})")

concrete functions traced: 2
final batch loss : 0.0066
learned: y = 2.998x + 1.977   (target: y = 3.0x + 2.0)


## 6. Gotchas & Pitfalls

- **Tensors are immutable; only `tf.Variable` is mutable.** You can't index-assign a `tf.Tensor` (`t[0] = 1` fails). Use `tf.Variable` for trainable state, or build new tensors with `tf.where`/`tf.tensor_scatter_nd_update`.
- **`tf.function` re-tracing.** Calling a `@tf.function` with Python scalars or new shapes/dtypes triggers a fresh trace each time — silently slow. Pass `tf.Tensor`s with stable shapes; use `input_signature` to pin it. A retrace warning in logs is the tell.
- **Python side effects don't run in graph mode.** Inside `@tf.function`, a bare `print()` runs only during tracing, not on every call; appending to Python lists won't behave as expected. Use `tf.print()` and TF ops for in-graph effects.
- **The tape only watches `tf.Variable`s by default.** To differentiate w.r.t. a constant `tf.Tensor`, call `tape.watch(x)` inside the block, or you'll get `None` gradients.
- **A non-persistent tape is consumed by the first `gradient()` call.** Need multiple gradients from one forward pass? `tf.GradientTape(persistent=True)` (and `del tape` when done).
- **`None` gradients.** Usually means the path from loss to variable left the tape (a NumPy op, an `int` cast, a detached value) or the variable wasn't used in the computation. Keep the whole forward pass in TF ops inside the tape.
- **dtype surprises.** TF is stricter than NumPy: mixing `float32` and `float64`, or feeding `int` where `float` is expected, raises instead of silently upcasting. Be explicit (`tf.cast`).
- **`model.predict()` vs calling the model.** `model(x)` is eager and fast for small/single inputs; `model.predict(x)` builds a batched graph loop and is for large datasets. Mixing them up gives confusing latency.
- **GPU memory is grabbed greedily.** TF allocates almost all GPU memory on startup by default. Use `tf.config.experimental.set_memory_growth(gpu, True)` when sharing a GPU.
- **Keras 3 is multi-backend now.** `import keras` may run on JAX/PyTorch depending on `KERAS_BACKEND`; `tf.keras` always targets TF. Know which you're importing. See [Keras](keras.ipynb).

## 7. When to Use vs Alternatives

| Tool | Use it when | Trade-off vs TensorFlow |
|------|-------------|-------------------------|
| **TensorFlow** | Production & on-device deployment (Serving / TFLite / TF.js), TPU targets, Google-ecosystem stacks | Smaller *research* mindshare today; lower-level API is more verbose than PyTorch |
| **[Keras](keras.ipynb)** | You want `model.fit()` and don't need the raw tape; fast model prototyping | It's the high-level API *on top of* TF (and now other backends) — less control for custom training |
| **[PyTorch](pytorch.ipynb)** | Research, fine-tuning open-weights models, the Hugging Face ecosystem | Most new research/models are PyTorch-first; TF's edge is deployment & TPUs |
| **[JAX/Flax](jax-flax.ipynb)** | Functional purity, composable `jit`/`vmap`/`grad`, large-scale TPU research | Steeper curve, smaller ecosystem; overlaps TF on TPUs but functional-style |
| **[scikit-learn](scikit-learn.ipynb)** | Classical ML — trees, linear/logistic, SVM, clustering on tabular data | Not deep learning; no autodiff/accelerator training of nets |
| **NumPy** | Pure array math, no learning/gradients | No autodiff, no GPU/TPU, no nn layers |

**Rule of thumb:** shipping to mobile/edge/browser/TPU, or already in a Google stack → TensorFlow (drive it through Keras). Research or consuming the open-weights ecosystem → PyTorch. TPU-scale functional research → JAX. Tabular/classical → scikit-learn.

## 8. Resources

- **Official docs & guides** — <https://www.tensorflow.org/guide> (eager, `tf.function`, autodiff, `tf.data` all covered well).
- **Install guide** — <https://www.tensorflow.org/install> (CPU/GPU wheels, Apple-Silicon `tensorflow-metal`).
- **`tf.function` / better performance** — <https://www.tensorflow.org/guide/function> (tracing, retracing, AutoGraph — the perf model you must internalize).
- **Autodiff with `GradientTape`** — <https://www.tensorflow.org/guide/autodiff> (watching, persistent tapes, higher-order gradients).
- **`tf.data` input pipelines** — <https://www.tensorflow.org/guide/data> (map/batch/shuffle/prefetch, performance).
- **Deploy** — TF Serving <https://www.tensorflow.org/tfx/guide/serving> · LiteRT/TFLite <https://ai.google.dev/edge/litert> · TF.js <https://www.tensorflow.org/js>.
- **Related notebooks in this library:** [Keras](keras.ipynb), [PyTorch](pytorch.ipynb), [JAX/Flax](jax-flax.ipynb), [scikit-learn](scikit-learn.ipynb).